<a href="https://colab.research.google.com/github/NVIDIA-NeMo/DataDesigner/blob/main/docs/colab_notebooks/8-generating-word-documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Word Documents with Data Designer

Data Designer generates rows. Your document pipeline wants `.docx` files.

The `data-designer-docx` plugin closes that gap. You describe the dataset as usual, add one processor,
and every row comes out as a real Word document — headings, tables, footers, the lot.

Install it, then seven short steps. Let's go.


### ⚡ Colab Setup

Run the cells below to install the dependencies and set up the API key. If you don't have an API key, you can generate one from [build.nvidia.com](https://build.nvidia.com).

This notebook uses the [`data-designer-docx`](https://github.com/NVIDIA-NeMo/DataDesignerPlugins/tree/main/plugins/data-designer-docx) plugin, which Data Designer discovers automatically once it is installed.


In [ ]:
%%capture
# data-designer-docx is not on PyPI yet; install it from the plugins repository.
# Once it is released this becomes: !pip install -U data-designer data-designer-docx
!pip install -U data-designer \
  "data-designer-docx @ git+https://github.com/NVIDIA-NeMo/DataDesignerPlugins.git#subdirectory=plugins/data-designer-docx"

In [ ]:
import getpass
import os

from google.colab import userdata

try:
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
except userdata.SecretNotFoundError:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")

## Step 0 · About the plugin

The `.docx` writer is not part of Data Designer. It is a separate plugin,
[`data-designer-docx`](https://github.com/NVIDIA-NeMo/DataDesignerPlugins/tree/main/plugins/data-designer-docx),
published in the NeMo Data Designer Plugins repository — the setup cell above already installed it.

Two things to know about plugin installs:

- **Nothing else is needed.** No registration call, no config entry. The package declares a
  `data_designer.plugins` entry point and Data Designer discovers it automatically.

- **Install before the kernel imports Data Designer.** Entry points are read from installed package
  metadata at import time, so a plugin installed into an already-running kernel is not seen until you
  restart the runtime.

Running locally instead of in Colab? Same install, any package manager:

```bash
pip install data-designer-docx
```


## Step 1 · Check the plugin is there

Data Designer finds plugins through installed package metadata, so this is really a check that the install
worked. If the list comes back empty, re-run the setup cell and restart the runtime.


In [ ]:
from data_designer.plugins.plugin import PluginType
from data_designer.plugins.registry import PluginRegistry

print("processor plugins:", PluginRegistry().get_plugin_names(PluginType.PROCESSOR))

## Step 2 · Imports and your API key

`WordDocument` comes from the plugin. It describes the *shape* of a document — title, summary, sections,
a table — and we will hand it straight to the LLM in a moment.

Note where it comes from: the schema ships **with the plugin**, because it is the contract between the
LLM and the renderer, and the two halves belong in the same package.


In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

from data_designer_docx.config import DocxProcessorConfig
from data_designer_docx.schema import WordDocument

data_designer = DataDesigner(artifact_path="./artifacts")

## Step 3 · Say what documents you want

Three samplers. They give every document an ID, a company, and a type — and because those values live in
the dataset, your finished corpus is labelled without you annotating anything.


In [ ]:
MODEL_ALIAS = "doc-writer"
MODEL_ID = "nvidia/nemotron-3-super-120b-a12b"

config_builder = dd.DataDesignerConfigBuilder(
    model_configs=[
        dd.ModelConfig(
            alias=MODEL_ALIAS,
            model=MODEL_ID,
            provider="nvidia",
            # A whole document in one call is a long generation. Give it room.
            inference_parameters=dd.ChatCompletionInferenceParams(temperature=0.9, max_tokens=8192),
        )
    ]
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="doc_id",
        sampler_type=dd.SamplerType.UUID,
        params=dd.UUIDSamplerParams(prefix="POL-", short_form=True, uppercase=True),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="company",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Northwind Diagnostics", "Cobalt Ridge Financial", "Halden Biopharma"]
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="doc_type",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Remote Work Policy",
                "Incident Response Runbook",
                "Vendor Onboarding Procedure",
                "Travel and Expense Policy",
            ]
        ),
    )
)

## Step 4 · Write the document

One LLM column does the writing. The important bit is `output_format=WordDocument`: instead of asking for
prose and parsing it afterwards, we ask for the document's *structure* and let the model fill it in.

That is why no `.docx` parsing code appears anywhere in this notebook.


In [ ]:
config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="document",
        model_alias=MODEL_ALIAS,
        output_format=WordDocument,
        prompt=(
            "Write an internal {{ doc_type }} for {{ company }} (document ID {{ doc_id }}).\n\n"
            "Write like a real corporate policy: flat, procedural, no marketing language. "
            "Sections should be specific to a {{ doc_type }} — scope, roles, the actual procedure, "
            "exceptions — not filler like 'Introduction'. "
            "The key_data table should hold concrete facts: thresholds, timeframes, who does what."
        ),
    )
)

## Step 5 · Turn each row into a `.docx`

Here it is. One processor, and the pipeline now writes Word files.

The templates are rendered per row, so anything in the dataset can go into the filename, the front-matter
table, or the footer.


In [ ]:
config_builder.add_processor(
    DocxProcessorConfig(
        name="word-documents",
        document_column="document",
        filename_template="{{ doc_id }}-{{ doc_type }}.docx",
        metadata_columns={"Document ID": "{{ doc_id }}", "Company": "{{ company }}"},
        footer_template="{{ company }} · {{ doc_id }}",
    )
)

data_designer.validate(config_builder)

## Step 6 · Preview one, then make ten

Always preview first. It is one document and one API call, and it tells you whether the prompt is landing
before you pay for a hundred.

Notice `docx_path` in the output — the processor writes it back into the dataset, so rows and files stay
joined.


In [ ]:
preview = data_designer.preview(config_builder, num_records=1)

preview.dataset[["doc_id", "company", "doc_type", "docx_path"]]

In [ ]:
results = data_designer.create(config_builder, num_records=10, dataset_name="policy-documents-plugin")

dataset = results.load_dataset()
dataset[["doc_id", "company", "doc_type", "docx_path"]]

## Step 7 · Open one

`docx_path` is relative to the dataset folder, so the whole thing stays portable. Double-click the file to
open it in Word — or read it back here, the way a downstream pipeline would.


In [ ]:
from docx import Document

document_path = results.artifact_storage.base_dataset_path / dataset["docx_path"].iloc[0]
rendered = Document(str(document_path))

print(f"{document_path.name}\n")
for paragraph in rendered.paragraphs:
    if paragraph.style.name.startswith(("Title", "Heading")):
        print(f"  {paragraph.text}")

print(f"\nfooter: {rendered.sections[0].footer.paragraphs[0].text}")
print(f"folder: {document_path.parent}")

## That's it 🎉

Ten real Word documents, and a dataset that knows which row produced which file.

Things you can change from here, all in that one `DocxProcessorConfig`:

| Want to... | Use |
| --- | --- |
| Apply your corporate styles | `template_path="brand/corporate-template.docx"` |
| Fill in Word's author/category fields | `core_property_columns={"author": "{{ owner }}"}` |
| Change where files land | `output_subdir="documents"` |
| Drop the `1.` `2.` numbering | `number_sections=False` |

And when you want more control over the documents themselves, edit `WordDocument` in
`src/data_designer_docx/schema.py`. The field descriptions in that file are sent to the model as part of
the prompt, so they are the fastest lever you have.

**Next:**

- [Plugins overview](https://docs.nvidia.com/nemo/datadesigner/plugins/overview) — writing your own

- [Processors](https://docs.nvidia.com/nemo/datadesigner/concepts/processors) — the built-in ones
